# Model Optimization: WANDA Pruning

In this notebook, we'll apply WANDA (Weight ANalysis for Deep leArning) pruning techniques to our models using distributed processing. This is an advanced pruning technique that considers both weight magnitudes and activation statistics when deciding which weights to prune.

## What is WANDA Pruning?

WANDA pruning is an advanced technique that improves upon traditional magnitude-based pruning methods by incorporating activation statistics. While standard pruning methods like L1 unstructured pruning only look at the absolute values of weights, WANDA considers how those weights interact with activations during inference.

### Key Differences from Standard Pruning:
- **Activation-Aware**: Considers both weight magnitudes and activation statistics
- **Better Accuracy Preservation**: Tends to maintain model accuracy better than simple magnitude pruning
- **Calibration Data**: Uses a small set of sample inputs to collect activation statistics
- **Importance Scoring**: Calculates importance as weight magnitude × activation magnitude

## 1. Import Dependencies

In [ ]:
import json
import time
import pandas as pd
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.pytorch.processing import PyTorchProcessor
import time
from IPython.display import clear_output

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")

## 3. Load Model Information and Previous Pruning Results

In [ ]:
# Try to load model information from file
try:
    with open('model_info.json', 'r') as f:
        model_info = json.load(f)
    print(f"Loaded information for {len(model_info)} models")
except FileNotFoundError:
    print("model_info.json not found. Using default model information.")
    model_info = {
        "sentiment-analysis": {
            "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
            "task": "text-classification",
            "hub_model_id": "distilbert-base-uncased-finetuned-sst-2-english",
            "s3_uri": f"s3://{S3_BUCKET}/models/distilbert-base-uncased-finetuned-sst-2-english"
        }
    }

# Try to load standard pruned metrics for comparison
try:
    with open('pruned-metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded standard pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned-metrics.json not found. Will proceed without standard pruning metrics for comparison.")
    pruned_metrics = {}

## 4. Create WANDA Pruning Script

In this section, we'll create a Python script that performs WANDA pruning. This script will be executed on the SageMaker Processing instances.

In [ ]:
# Check if the wanda_pruning_scripts directory exists, if not create it
import os
if not os.path.exists('wanda_pruning_scripts'):
    os.makedirs('wanda_pruning_scripts')
    print("Created wanda_pruning_scripts directory")
else:
    print("wanda_pruning_scripts directory already exists")

## 5. Launch Distributed WANDA Pruning Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform WANDA pruning. Each model will be processed in a separate job, allowing for parallel processing.

In [ ]:
# Define the instance type to use for pruning
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="2.0.0",
    py_version="py310",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="wanda-pruning",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch WANDA pruning jobs for all models in parallel
job_names = []  # List to store all job names
job_output_paths = {}
s3_client = boto3.client('s3')

# First, prepare all the job configurations
job_configs = {}
print("Preparing WANDA pruning jobs for all models...")

for model_key in model_info.keys():
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define the output path
    output_path = f's3://{S3_BUCKET}/optimization/outputs/{model_key}-wanda-pruned'
    job_output_paths[model_key] = output_path
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            output_name='wanda-pruned-model',
            source='/opt/ml/processing/output',
            destination=output_path
        )
    ]
    
    # Store the job configuration
    job_configs[model_key] = {
        'inputs': inputs,
        'outputs': outputs,
        'arguments': [
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--pruning-amount', '0.3',
            '--calibration-samples', '32'  # Number of samples to use for activation statistics
        ]
    }
    print(f"Prepared job configuration for {model_key}")

# Now launch all jobs in parallel
print("\nLaunching all WANDA pruning jobs in parallel...")
for model_key, config in job_configs.items():
    try:
        # Create a unique job name with timestamp to avoid conflicts
        timestamp = int(time.time())
        job_name = f"wanda-pruning-{model_key}-{timestamp}"
        
        # Run the processing job with the unique name
        processor.run(
            code='wanda_pruning_script.py',
            source_dir='wanda_pruning_scripts',
            inputs=config['inputs'],
            outputs=config['outputs'],
            arguments=config['arguments'],
            wait=False,  # Don't wait for the job to complete before continuing
            job_name=job_name  # Explicitly set the job name
        )
        
        # Store the job name for tracking
        job_names.append(job_name)
        print(f"Launched job for {model_key}: {job_name}")
    except Exception as e:
        print(f"Error launching job for {model_key}: {e}")

print("\nAll jobs launched. You can monitor their progress in the SageMaker console.")

In [ ]:
# Monitor job status
from sagemaker.processing import ProcessingJob

# Function to check if all jobs are complete
def are_all_jobs_complete(job_names, sagemaker_session):
    all_complete = True
    job_statuses = {}
    
    # Create a SageMaker client for API calls
    sagemaker_client = boto3.client('sagemaker')
    
    for job_name in job_names:
        try:
            # Use the SageMaker client to describe the processing job
            response = sagemaker_client.describe_processing_job(
                ProcessingJobName=job_name
            )
            status = response['ProcessingJobStatus']
            job_statuses[job_name] = status
            
            if status in ['InProgress', 'Stopping']:
                all_complete = False
        except Exception as e:
            job_statuses[job_name] = f"Error: {str(e)}"
            # Consider jobs with errors as complete to avoid infinite loops
            
    return all_complete, job_statuses

# Poll for job completion
print("Waiting for all jobs to complete...")
while True:
    all_complete, job_statuses = are_all_jobs_complete(job_names, sagemaker_session)
    
    # Clear previous output
    clear_output(wait=True)
    
    # Print current status
    print("Current job statuses:")
    for job_name, status in job_statuses.items():
        print(f"Job {job_name}: {status}")
    
    if all_complete:
        print("All jobs completed!")
        break
    
    print("Waiting for jobs to complete... Will check again in 60 seconds.")
    time.sleep(60)  # Check every minute

print("\nAll jobs have completed or failed.")

## 7. Collect Results

Now that the WANDA pruning jobs are complete, we'll collect and combine the results from each job.

In [ ]:
# Download and combine results
wanda_pruned_metrics = {}
sagemaker_client = boto3.client("sagemaker")

for model_key in model_info.keys():
    # Check if we have a job for this model
    if model_key not in job_output_paths:
        print(f"No output path found for {model_key}, skipping metrics collection")
        continue
        
    # Download metrics file
    try:
        # Use the saved output path
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}-wanda-pruned/wanda-pruned-metrics.json',
            f'temp_{model_key}-wanda-pruned-metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}-wanda-pruned-metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        wanda_pruned_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('wanda-pruned-metrics.json', 'w') as f:
    json.dump(wanda_pruned_metrics, f, indent=2)

print(f"\nSaved WANDA pruned metrics for {len(wanda_pruned_metrics)} models to wanda-pruned-metrics.json")

## 8. Compare WANDA Pruning with Standard Pruning

Now we'll compare the results of WANDA pruning with standard pruning to see the differences in model size, inference time, and accuracy.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

# Function to get total size of objects with a prefix from S3
def get_total_size(bucket, prefix):
    total_size = 0
    paginator = s3_client.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                total_size += obj['Size']
    return total_size

for model_key in wanda_pruned_metrics.keys():
    standard_pruned = pruned_metrics.get(model_key, {})
    wanda_pruned = wanda_pruned_metrics[model_key]
    
    # Get original model size from S3
    original_model_prefix = model_info[model_key]['s3_uri'].replace(f"s3://{S3_BUCKET}/", "")
    original_size = get_total_size(S3_BUCKET, original_model_prefix)
    original_size_mb = original_size / (1024 * 1024)
    
    # Get WANDA pruned model size
    wanda_model_size_mb = wanda_pruned.get('model_size', 0)
    
    # Calculate size reduction percentage
    wanda_size_reduction = ((original_size_mb - wanda_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
    
    # Prepare data for this model
    model_data = {
        'Model': wanda_pruned['model_name'],
        'Original Size (MB)': f"{original_size_mb:.2f}",
        'WANDA Pruning Amount': f"{wanda_pruned['pruning_amount'] * 100:.1f}%",
        'WANDA Pruned Size (MB)': f"{wanda_model_size_mb:.2f}",
        'WANDA Size Reduction (%)': f"{wanda_size_reduction:.2f}",
        'WANDA Sparsity (%)': f"{wanda_pruned.get('sparsity', 0):.2f}"
    }
    
    # Add standard pruning data if available
    if standard_pruned:
        standard_model_size_mb = standard_pruned.get('model_size', 0)
        standard_size_reduction = ((original_size_mb - standard_model_size_mb) / original_size_mb) * 100 if original_size_mb > 0 else 0
        
        model_data.update({
            'Standard Pruning Method': standard_pruned.get('pruning_method', 'N/A'),
            'Standard Pruning Amount': f"{standard_pruned.get('pruning_amount', 0) * 100:.1f}%",
            'Standard Pruned Size (MB)': f"{standard_model_size_mb:.2f}",
            'Standard Size Reduction (%)': f"{standard_size_reduction:.2f}",
            'Standard Sparsity (%)': f"{standard_pruned.get('sparsity', 0):.2f}"
        })
    
    comparison_data.append(model_data)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 9. Visualize Comparison

Let's visualize the comparison between WANDA pruning and standard pruning.

In [ ]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the visualization style
sns.set(style="whitegrid")
plt.figure(figsize=(12, 6))

# Create a DataFrame for visualization
viz_data = []

for model_key in wanda_pruned_metrics.keys():
    if model_key in pruned_metrics:
        model_name = wanda_pruned_metrics[model_key]['model_name']
        
        # Add standard pruning data
        viz_data.append({
            'Model': model_name,
            'Pruning Method': 'Standard',
            'Size Reduction (%)': ((original_size_mb - pruned_metrics[model_key].get('model_size', 0)) / original_size_mb) * 100 if original_size_mb > 0 else 0,
            'Sparsity (%)': pruned_metrics[model_key].get('sparsity', 0) * 100
        })
        
        # Add WANDA pruning data
        viz_data.append({
            'Model': model_name,
            'Pruning Method': 'WANDA',
            'Size Reduction (%)': ((original_size_mb - wanda_pruned_metrics[model_key].get('model_size', 0)) / original_size_mb) * 100 if original_size_mb > 0 else 0,
            'Sparsity (%)': wanda_pruned_metrics[model_key].get('sparsity', 0) * 100
        })

# Create DataFrame for visualization
viz_df = pd.DataFrame(viz_data)

# Create the bar chart
ax = sns.barplot(x="Model", y="Size Reduction (%)", hue="Pruning Method", data=viz_df)

# Customize the chart
plt.title("Size Reduction Comparison: Standard vs. WANDA Pruning")
plt.xlabel("Model")
plt.ylabel("Size Reduction (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Pruning Method")
plt.tight_layout()

# Show the chart
plt.show()

## 10. Next Steps

Now that we've applied WANDA pruning to our models and compared it with standard pruning, we'll explore knowledge distillation in the next notebook to create even smaller, more efficient models.

### What We've Learned:
- How WANDA pruning differs from standard pruning by considering activation statistics
- How to implement WANDA pruning using SageMaker Processing jobs
- How WANDA pruning compares to standard pruning in terms of model size reduction and sparsity
- How to run multiple processing jobs in parallel for faster experimentation

### What's Next - Knowledge Distillation:
Knowledge distillation is a technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. This approach can lead to even more significant size reductions while maintaining good performance.